<a href="https://colab.research.google.com/github/codingahahahhaah/compling/blob/main/fine_tuning_hw_ipynb_%D0%B5%D1%81%D0%BB%D0%B8_%D0%BF%D1%80%D0%B5%D0%B4%D1%8B%D0%B4%D1%83%D1%89%D0%B8%D0%B9_%D1%84%D0%B0%D0%B9%D0%BB_%D0%BD%D0%B5_%D0%BE%D1%82%D0%BA%D1%80%D1%8B%D0%BB%D1%81%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели


In [10]:
# Устанавливаем все технические штуки
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU доступен: True
Тип GPU: Tesla T4


In [11]:
# Загружаем датасет
dataset = load_dataset("ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

Датасет загружен. Train: 120000, Test: 7600


In [12]:
# Загружаем модель и токенизатор
model_name = "bert-base-uncased"  # с такой на семинаре не работали, поэтому взяла её
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# Готовим данные
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000))

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [26]:
# Настраиваем обучение
training_args = TrainingArguments(
    output_dir="./ag_news_model", # чтобы оно сразу сохранялось в нужную папку
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

In [14]:
# Вычисляем метрики
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [16]:
# Обучаем
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [17]:
# Выводим accurancy
eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")

Epoch,Training Loss,Validation Loss,Accuracy
0,No log,0.294933,0.898800



Evaluation results: {'eval_loss': 0.2949328124523163, 'eval_accuracy': 0.8988}


In [25]:
# Тестируем модель на новостях
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

test_texts = [
    "Neil Simpson wins first Great Britain medal at Winter Paralympics with skiing silver",
    "Elevated Energy Prices Add to Fed’s Dilemma on Interest Rates",
    "YouTube Adds Tool to Help Public Figures Report Fake Videos"
]

label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

for text in test_texts:
    result = classifier(text)[0]
    pred_class_id = int(result['label'].split('_')[-1])  # 'LABEL_2' → 2
    pred_class = label_names[pred_class_id]
    print(f"Text: {text}\nClass: {pred_class} ({pred_class_id}), Score: {result['score']:.4f}\n")

Text: Neil Simpson wins first Great Britain medal at Winter Paralympics with skiing silver
Class: Sports (1), Score: 0.9044

Text: Elevated Energy Prices Add to Fed’s Dilemma on Interest Rates
Class: Business (2), Score: 0.9612

Text: YouTube Adds Tool to Help Public Figures Report Fake Videos
Class: Sci/Tech (3), Score: 0.9304

